## Ingest from Ebay API and Load into bronze layer in duckdb

In [ ]:
import polars as pl
from polars import col

import duckdb

from dotenv import load_dotenv
import os
import requests
import base64

from datetime import date

### Config

In [ ]:
# DuckDB Connection
con = duckdb.connect("../burger.db")

# Environment
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

### Pull from Ebay API
#### Get Token

In [ ]:
# Get Token
client_id = os.getenv('EBAY_CLIENT_ID')
client_secret = os.getenv('EBAY_SECRET')

def get_ebay_token():
    auth_string = f"{client_id}:{client_secret}"
    b64_encoded = base64.b64encode(auth_string.encode()).decode()
    
    url = 'https://api.ebay.com/identity/v1/oauth2/token' # Use api.sandbox.ebay.com for testing
    headers = {
        'Authorization': f'Basic {b64_encoded}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'client_credentials',
        'scope': 'https://api.ebay.com/oauth/api_scope'
    }
    
    response = requests.post(url, headers=headers, data=data)
    response_data = response.json()
    
    return response_data.get('access_token')

# Get the token to use in subsequent requests
token = get_ebay_token()


### Get Data

In [ ]:
# Search for Burgerchu Cards
def search_ebay_items(access_token, query):
    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    params = {
        'q': query,
        'limit': 50
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        return f"Error {response.status_code}: {response.text}"

results = search_ebay_items(token, "2025 POKEMON JAPANESE M-P PROMO MCDONALD'S #020 PIKACHU PSA 10")

#### Create Polars Dataframe from Results

In [ ]:
def items_to_df(results: dict) -> pl.DataFrame:
    rows = []
    for item in results.get("itemSummaries", []):
        price = item.get("price") or {}
        seller = item.get("seller") or {}

        opts = item.get("shippingOptions") or []
        ship = opts[0]["shippingCost"] if opts else {}
        ship_cost = float(ship["value"]) if ship.get("value") is not None else None

        rows.append({
            "item_id": item.get("itemId"),
            "title": item.get("title"),
            "condition": item.get("condition"),
            "price": float(price["value"]) if price.get("value") is not None else None,
            "currency": price.get("currency"),
            "shipping_cost": ship_cost,
            "total_cost": (float(price["value"]) + ship_cost)
                          if price.get("value") is not None and ship_cost is not None
                          else None,
            "seller_feedback_pct": seller.get("feedbackPercentage"),
            "item_url": item.get("itemWebUrl"),
        })
    return pl.DataFrame(rows)

df = items_to_df(results)

#### Add in Date

In [ ]:
df = df.with_columns(pl.lit(date.today()).alias("loaded_at"))
# display(df)

#### Load into DuckDB

In [ ]:
con.execute("INSERT INTO bronze SELECT * FROM df")

con.close()